In [1]:
import sys
sys.path.append("/home/onyxia/work/skill_match/skill_match")

In [ ]:
import kagglehub
import shutil
from pathlib import Path

# Télécharger le dataset depuis Kaggle
path_resume = kagglehub.dataset_download("hadikp/resume-data-pdf")
path_jobs = kagglehub.dataset_download("ravindrasinghrana/job-description-dataset")
path_cv = kagglehub.dataset_download("suriyaganesh/resume-dataset-structured")
# Destination dans ton projet
destination_resume = Path("~/work/skill_match/data/resume-data-pdf").expanduser()
destination_job = Path("~/work/skill_match/data/job-description-dataset").expanduser()
destination_cv = Path("~/work/skill_match/data/resume-dataset-structured").expanduser()

# Copier le dataset dans data
shutil.copytree(path_resume, destination_resume, dirs_exist_ok=True)
shutil.copytree(path, destination_job, dirs_exist_ok=True)
shutil.copytree(path_cv, destination_cv, dirs_exist_ok=True)


print("Dataset copié dans :", destination_cv)
print("Dataset copié dans :", destination_resume)
print("Dataset copié dans :", destination_job)

KeyboardInterrupt: 

In [2]:
import pandas as pd
df = pd.read_csv("/home/onyxia/work/skill_match/data/cv_54k_reconstructed.csv")
df.head()

,person_id,name,email,phone,linkedin,abilities,skills,education,experience,cv_text
0,1,Database Administrator,NaN,NaN,NaN,"['Installation and Building Server', 'Running ...","['Database administration', 'Database', 'Ms sq...",['Bachelor of Science at Lead City University ...,['Database Administrator at Family Private Car...,"Skills: Database administration, Database, Ms ..."
1,2,Database Administrator,NaN,NaN,NaN,"['database management systems administration',...","['sql server management studio', 'visual studi...",['bsc in computer science at lagos state unive...,['Database Administrator at Intercontinental R...,"Skills: sql server management studio, visual s..."
2,3,Oracle Database Administrator,NaN,NaN,NaN,['Over 4+ years of Experience as Architecture'...,"['DATABASES', 'ORACLE (4 years)', 'ORACLE 10G'...",['Master of Computer Applications in Science a...,['Oracle Database Administrator at Cognizant (...,"Skills: DATABASES, ORACLE (4 years), ORACLE 10..."
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,"['SQL management', 'PostgresSQL', 'Oracle', 'M...",['Maintain multiple database environments (Red...,['Bachelor in Computer Science at University o...,['Amazon Redshift Administrator and ETL Develo...,Skills: Maintain multiple database environment...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,"['Scrum Master', 'Agile software development',...","['Scrum', 'Agile software development', 'Produ...",['nan at Virginia Commomwealth University (08/...,['Scrum Master at Quest Technologies (10/2015 ...,"Skills: Scrum, Agile software development, Pro..."


In [3]:
import pandas as pd

from fonctions_nlp import (
    preprocess_text, prepare_jobs, load_cv_folder,load_single_cv,load_cv_csv,
    train_doc2vec, compute_matching_score,
    init_ner_pipeline, extract_skills_list_clean,create_tagged_documents
)


jobs = pd.read_csv("/home/onyxia/work/skill_match/data/job-description-dataset/job_descriptions.csv")
jobs_sample = jobs.sample(n=54933, random_state=42)

jobs_sample.head()

#jobs = pd.read_csv("/home/onyxia/work/skill_match/data/job_dataset.zip",compression="zip")
#jobs.head()



,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
108318,1017340707950150,5 to 10 Years,BBA,$55K-$84K,Panama City,Panama,8.5379,-80.7821,Contract,93242,...,242.271.4459,Procurement Manager,Supplier Diversity Manager,The Muse,Promote diversity and inclusion in the supply ...,"{'Transportation Benefits, Professional Develo...",Supplier diversity programs Diversity and incl...,Promote supplier diversity initiatives and inc...,RWE AG,"{""Sector"":""Energy"",""Industry"":""Energy - Utilit..."
7787,2421048253959975,0 to 12 Years,MBA,$61K-$108K,Tunis,Tunisia,33.8869,9.5375,Part-Time,18411,...,579.442.3566,Architectural Designer,Architectural Drafter,Idealist,Architectural Drafters assist architects and e...,"{'Employee Assistance Programs (EAP), Tuition ...",Architectural drafting AutoCAD 2D and 3D model...,Prepare detailed architectural drawings and pl...,Asian Paints,"{""Sector"":""Consumer Goods"",""Industry"":""Paints ..."
1237496,1822636506606589,0 to 11 Years,M.Com,$57K-$82K,Harare,Zimbabwe,-19.0154,29.1549,Full-Time,120621,...,858-776-8996,Art Teacher,Art Education Coordinator,ZipRecruiter,An Art Education Coordinator plans and manages...,"{'Employee Referral Programs, Financial Counse...",Art education curriculum Program development T...,"Coordinate art education programs, curriculum ...",Laboratory Corp. of America,"{""Sector"":""Healthcare Services"",""Industry"":""He..."
55757,3068000579894602,5 to 12 Years,B.Com,$56K-$95K,Tirana,Albania,41.1533,20.1683,Temporary,128908,...,938.587.7586x35852,Environmental Consultant,Environmental Impact Analyst,Internships.com,Environmental Impact Analysts assess the envir...,"{'Transportation Benefits, Professional Develo...",Environmental impact analysis Data collection ...,Assess the environmental impact of projects an...,Massachusetts Mutual Life Insurance,"{""Sector"":""Insurance"",""Industry"":""Insurance: L..."
818970,1747904829392680,4 to 13 Years,BCA,$58K-$122K,City of Baghdad,Iraq,33.2232,43.6793,Temporary,114717,...,(405)990-8581x57164,Art Teacher,Art Education Coordinator,LinkedIn,An Art Education Coordinator plans and manages...,"{'Employee Referral Programs, Financial Counse...",Art education curriculum Program development T...,"Coordinate art education programs, curriculum ...",Sartorius AG,"{""Sector"":""Lab Equipment"",""Industry"":""Life Sci..."


In [7]:
# on garde uniquement les colonnes dont on a besoin et on crée
# la colonne job_text qui regroupe toutes les informations pour une offre en un bloc
#jobs_precis

In [4]:
# On charge et on nettoie les jobs
jobs_precis = prepare_jobs(jobs_sample)

# pareil pour les cv
cv_df = load_cv_csv("/home/onyxia/work/skill_match/data/cv_54k_reconstructed.csv")

# On tag les documents pour doc2vec
tagged_jobs, tagged_cvs = create_tagged_documents(jobs_precis, cv_df)

In [5]:

print(f"Nombre de CV extraits : {len(cv_df)}")

Nombre de CV extraits : 54933


In [6]:
# Prétraiter le texte
cv_df['cv_text_clean'] = cv_df['cv_text'].apply(preprocess_text)

# Afficher un aperçu
cv_df[['cv_text', 'cv_text_clean']].head()

,cv_text,cv_text_clean
0,skills database administration database ms sql...,skills database administration database ms sql...
1,skills sql server management studio visual stu...,skills sql server management studio visual stu...
2,skills databases oracle years oracle g sql lin...,skills databases oracle years oracle g sql lin...
3,skills maintain multiple database environments...,skills maintain multiple database environments...
4,skills scrum agile software development produc...,skills scrum agile software development produc...


In [7]:
all_tagged = tagged_jobs + tagged_cvs
model = train_doc2vec(all_tagged, vector_size=100, window=5, min_count=3, workers=8, epochs=20)
model.save("/home/onyxia/work/skill_match/models/doc2vec_model_cv_job")
print("Modèle Doc2Vec entraîné et sauvegardé !")

Taille du vocabulaire : 61176
Modèle Doc2Vec entraîné et sauvegardé !


In [8]:
# Pour évaluer la qualité du modèle le prof a demande de créer un jeu de test annoté c'est à dire on dit si
# le matching entre un cv et une offre est pertinente et pas pertinent . il a dit de le faire pour top10 cv
# un truc comme ça

In [9]:
# Construire cv_texts_clean depuis cv_df
cv_texts_clean = dict(zip(cv_df.index.astype(str), cv_df['cv_text']))

# Top 10 jobs pour le test
job_texts_test = jobs_precis['job_text'][:10]

# Inférer vecteurs jobs
job_vectors = [model.infer_vector(j.split()) for j in job_texts_test]

# Inférer vecteurs CV (on prend les 10 premiers pour ne pas exploser la mémoire)
cv_texts_clean_sample = dict(list(cv_texts_clean.items())[:10])
cv_vectors = [model.infer_vector(cv.split()) for cv in cv_texts_clean_sample.values()]
cv_names   = list(cv_texts_clean_sample.keys())

In [10]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import pandas as pd

# ==============================================
# 1. MATRICE DE SIMILARITÉ (jobs x CVs)
# ==============================================
similarity_matrix = cosine_similarity(job_vectors, cv_vectors)

# Afficher sous forme de DataFrame
sim_df = pd.DataFrame(
    similarity_matrix,
    index   = [f"Job_{i}" for i in range(len(job_vectors))],
    columns = cv_names
)
print(sim_df.round(3))

# ==============================================
# 2. HEATMAP
# ==============================================
fig = px.imshow(
    sim_df,
    text_auto = ".2f",
    color_continuous_scale = "Blues",
    title = "Matrice de similarité Jobs × CVs",
    labels = dict(x="CV", y="Job", color="Score")
)
fig.update_layout(
    paper_bgcolor = "#0a0f1a",
    plot_bgcolor  = "#0a0f1a",
    font          = dict(color="#e2e8f0"),
    title_font    = dict(size=14)
)
fig.show()

# ==============================================
# 3. MÉTRIQUES D'ÉVALUATION
# ==============================================
# Pour chaque job, quel CV est le mieux classé ?
print("\n--- Meilleur CV par job ---")
for i, job_scores in enumerate(similarity_matrix):
    best_cv_idx   = np.argmax(job_scores)
    best_cv_score = job_scores[best_cv_idx]
    print(f"Job_{i} → meilleur CV : {cv_names[best_cv_idx]} (score : {best_cv_score:.3f})")

# Score moyen global
print(f"\nScore moyen global    : {similarity_matrix.mean():.3f}")
print(f"Score max global      : {similarity_matrix.max():.3f}")
print(f"Score min global      : {similarity_matrix.min():.3f}")
print(f"Écart-type            : {similarity_matrix.std():.3f}")

# ==============================================
# 4. DISTRIBUTION DES SCORES
# ==============================================
scores_flat = similarity_matrix.flatten()

fig2 = px.histogram(
    x     = scores_flat,
    nbins = 30,
    title = "Distribution des scores de similarité",
    labels = dict(x="Score cosinus", y="Fréquence")
)
fig2.update_traces(marker_color="#22d3ee", marker_line_color="#0a0f1a", marker_line_width=1)
fig2.update_layout(
    paper_bgcolor = "#0a0f1a",
    plot_bgcolor  = "#0a0f1a",
    font          = dict(color="#e2e8f0"),
    xaxis         = dict(gridcolor="#1a2744"),
    yaxis         = dict(gridcolor="#1a2744")
)
fig2.show()

           0      1      2      3      4      5      6      7      8      9
Job_0  0.378  0.309  0.379  0.260  0.485  0.432  0.437  0.187  0.317  0.551
Job_1  0.327  0.275  0.289  0.309  0.359  0.340  0.297  0.256  0.203  0.513
Job_2  0.240  0.255  0.287  0.152  0.423  0.537  0.448  0.178  0.282  0.602
Job_3  0.294  0.314  0.284  0.262  0.291  0.294  0.255  0.185  0.179  0.451
Job_4  0.329  0.333  0.346  0.279  0.447  0.492  0.464  0.226  0.248  0.633
Job_5  0.372  0.384  0.382  0.352  0.473  0.405  0.434  0.258  0.271  0.624
Job_6  0.331  0.305  0.348  0.277  0.492  0.447  0.442  0.272  0.323  0.593
Job_7  0.307  0.319  0.295  0.281  0.420  0.422  0.412  0.231  0.303  0.568
Job_8  0.347  0.299  0.320  0.349  0.453  0.366  0.377  0.244  0.273  0.571
Job_9  0.414  0.323  0.407  0.317  0.446  0.405  0.402  0.269  0.301  0.569



--- Meilleur CV par job ---
Job_0 → meilleur CV : 9 (score : 0.551)
Job_1 → meilleur CV : 9 (score : 0.513)
Job_2 → meilleur CV : 9 (score : 0.602)
Job_3 → meilleur CV : 9 (score : 0.451)
Job_4 → meilleur CV : 9 (score : 0.633)
Job_5 → meilleur CV : 9 (score : 0.624)
Job_6 → meilleur CV : 9 (score : 0.593)
Job_7 → meilleur CV : 9 (score : 0.568)
Job_8 → meilleur CV : 9 (score : 0.571)
Job_9 → meilleur CV : 9 (score : 0.569)

Score moyen global    : 0.357
Score max global      : 0.633
Score min global      : 0.152
Écart-type            : 0.106


In [11]:
# Regarder le contenu du CV 9 (le meilleur partout)
print(cv_df.iloc[9]['cv_text'][:500])



skills database oracle pl sql sql encryption rman linux shell scripting unix unix shell etl informatica bash scripting security capacity planning performance tuning oem cloning architecture dba oracle dba sql server database oracle pl sql sql linux unix rman datapump dbua data guard database oracle datapump dbua rman linux windows sql server abilities database design monitoring performance tuning migration upgrades backup recovery cloning replication security implementations analytical verbal wr


In [12]:
# Regarder le contenu du CV 3 (le pire partout)
print(cv_df.iloc[3]['cv_text'][:500])



skills maintain multiple database environments redshift rds in aws user management and group in redshift design and developed etl jobs to extract data from rds and load it in redshift matillion etl and python script experience working within the aws big data ecosystem s rds and redshift expertise in development of various reports and prepare the data for the utilization with tableau expertise query tuning and performance optimization and implementing work load management managing amazon redshift


In [13]:
# Comparer avec le Job_4 qui a le plus grand écart entre CVs
print(jobs_precis.iloc[4]['job_text'][:500])

art teacher an art education coordinator plans and manages art education programs curriculum development and educational outreach to promote art appreciation and learning within communities art education curriculum program development teacher training communication skills artistic expertise art education coordinator coordinate art education programs curriculum development and student assessments collaborate with other educators and schools organize art exhibitions and showcases bca sartorius ag


In [14]:
# le modèle classe correctement les CVs pertinents au-dessus des non pertinents
# cependant les scores absolus restent élevés même pour des paires incohérentes

In [15]:
from pathlib import Path
cv_test_folder = Path("/home/onyxia/work/skill_match/cv_test")
cv_file = list(cv_test_folder.glob("*.pdf"))[0]  # prend le premier PDF
cv_text = load_single_cv(cv_file)

In [16]:
import ipywidgets as widgets
from IPython.display import display

job_input = widgets.Text(
    value='',
    placeholder='Description du job',
    description='Job:',
)
display(job_input)

Text(value='', description='Job:', placeholder='Description du job')

In [17]:
job_text_clean = preprocess_text(job_input.value)
print(job_text_clean)

who s rollee rollee is building the next gen credit bureau to make financial services accessible for everyone we work with auto lenders banks insurance companies and many other businesses to streamline their underwriting process you can be an uber driver a tiktoker or thriving as a freelance consultant we make your income data accessible readable and in the proper context for lenders so you get what you deserve what s the role we are looking for a product data scientist intern to join our team during months you ll sit at the intersection of product and data ensuring that rollee s data is accurate reliable and truly valuable for our customers working with various datasets to extract meaningful information with a business approach running business qa on datasets we collect structuring rough data into exploitable features available through our api developing internal and external business intelligence models teaming up with product business and tech to implement data driven strategies you

In [18]:
from pathlib import Path
from fonctions_nlp import load_single_cv

cv_test_folder = Path("/home/onyxia/work/skill_match/cv_test")
cv_file = list(cv_test_folder.glob("*.pdf"))[0]  # prend le premier PDF
cv_texts_test = {cv_file.name: load_single_cv(cv_file)}

In [19]:
import plotly.graph_objects as go
from gensim.models import Doc2Vec
# Charger le modèle Doc2Vec
model = Doc2Vec.load("/home/onyxia/work/skill_match/models/doc2vec_model_cv_job")

# Calculer le score pour chaque CV de test et afficher le gauge
for fname, cv_text in cv_texts_test.items():
    score = compute_matching_score(model, job_text_clean, cv_text)
    print(f"{fname} → Score de matching : {score:.2f}")

    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=score * 100,  # convertir en %
        title={'text': f"Matching {fname}"},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': "darkblue"},
            'steps': [
                {'range': [0, 50], 'color': "red"},
                {'range': [50, 75], 'color': "yellow"},
                {'range': [75, 100], 'color': "green"}],
        }
    ))
    fig.show()

cv_english.pdf → Score de matching : 0.43


In [20]:
# on initialise le pipeline
ner_pipeline = init_ner_pipeline()

# on test un cv et une offre
job_text = job_text_clean
cv_text = cv_texts_test["cv_english.pdf"]

skills_in_job  = extract_skills_list_clean(job_text,  ner_pipeline)
skills_in_cv   = extract_skills_list_clean(cv_text,   ner_pipeline)
missing_skills = list(set(skills_in_job) - set(skills_in_cv))

print("Compétences détectées dans le job :", skills_in_job)
print("Compétences présentes dans le CV  :", skills_in_cv)
print("Compétences manquantes dans le CV :", missing_skills)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Compétences détectées dans le job : ['python', 'computer science', 'api', 'machine learning', 'sql', 'conversation', 'business intelligence', 'english', 'data science', 'git']
Compétences présentes dans le CV  : ['python', 'machine', 'raw', 'prototyping for', 'computer science', 'github com', 'data analysis', 'solving experience', 'machine learning', 'business', 'apis', 'power', 'english', 'sql', 'git', 'business intelligence', 'problem']
Compétences manquantes dans le CV : ['data science', 'conversation', 'api']


In [ ]:
# le prof a dit que ce modèle d'extraction était plutôt bon mais par exemple
# on pourrait évaluer sa qualité d'extraction
# c'est à dire extraire les compétences d'un cv via la bibliothèque python et faire une comparaison avec les compétences
# extraites via le modèle pré-entraîné comme ça on peut savoir si le modèle est adapté au projet

In [27]:
import spacy
from spacy.matcher import PhraseMatcher
from skillNer.general_params import SKILL_DB
from skillNer.skill_extractor_class import SkillExtractor

# Initialiser skillNer
nlp = spacy.load("en_core_web_lg")
skill_extractor = SkillExtractor(nlp, SKILL_DB, PhraseMatcher)

def extract_skills_skillner(text: str) -> list:
    annotations = skill_extractor.annotate(text)
    skills = [
        s['doc_node_value'].lower()
        for s in annotations['results']['full_matches']
    ] + [
        s['doc_node_value'].lower()
        for s in annotations['results']['ngram_scored']
        if s['score'] >= 0.7
    ]
    return list(set(skills))

# Extraction référence sur job et CV
skills_ref_job = extract_skills_skillner(job_text)
skills_ref_cv  = extract_skills_skillner(cv_text)

print("Référence job :", skills_ref_job)
print("Référence CV  :", skills_ref_cv)

loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...


/opt/python/lib/python3.13/site-packages/skillNer/utils.py:99: UserWarning: [W008] Evaluating Token.similarity based on empty vectors.
  vec_similarity = token1.similarity(token2)


Référence job : ['nice', 'datasets', 'api', 'streamline', 'analytics', 'sql', 'financial service', 'consultant', 'product manager', 'mathematics computer', 'read', 'diversity and inclusion', 'patchwork', 'english', 'statistics', 'git', 'data science', 'python', 'algorithms', 'computer science', 'statistic', 'people manager', 'underwriting', 'business intelligence']
Référence CV  : ['datasets', 'api', 'internal business', 'analytics', 'data structuring', 'r', 'dashboards', 'b', 'french', 'sql', 'automation', 'numpy', 'management', 'reliability', 'data processing', 'communication', 'ai apis', 'english', 'm', 'languages', 'raw datum', 'tableau', 'teamwork', 'power bi', 'com', 'predictive model', 'git', 'loan', 'programming', 'github', 'data science', 'problem solve', 'languages english', 'python', 'predictive modeling', 'algorithms', 'prototyping', 'data', 'computer science', 'scikit learn', 'mathematics statistics', 'collaborating', 'business intelligence', 'tensorflow', 'machine learn',

In [28]:
def evaluate_ner(predicted: list, reference: list) -> dict:
    predicted_set = set(predicted)
    reference_set = set(reference)

    tp = len(predicted_set & reference_set)
    fp = len(predicted_set - reference_set)
    fn = len(reference_set - predicted_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision"      : round(precision, 2),
        "recall"         : round(recall, 2),
        "f1"             : round(f1, 2),
        "tp"             : tp,
        "fp"             : fp,
        "fn"             : fn,
        "faux_positifs"  : list(predicted_set - reference_set),
        "manques"        : list(reference_set - predicted_set)
    }

# Calcul métriques
metrics_job = evaluate_ner(skills_in_job,  skills_ref_job)
metrics_cv  = evaluate_ner(skills_in_cv,   skills_ref_cv)

print("=== Évaluation NER — Job ===")
print(f"Précision      : {metrics_job['precision']}")
print(f"Rappel         : {metrics_job['recall']}")
print(f"F1             : {metrics_job['f1']}")
print(f"Faux positifs  : {metrics_job['faux_positifs']}")
print(f"Skills manqués : {metrics_job['manques']}")

print("\n=== Évaluation NER — CV ===")
print(f"Précision      : {metrics_cv['precision']}")
print(f"Rappel         : {metrics_cv['recall']}")
print(f"F1             : {metrics_cv['f1']}")
print(f"Faux positifs  : {metrics_cv['faux_positifs']}")
print(f"Skills manqués : {metrics_cv['manques']}")

=== Évaluation NER — Job ===
Précision      : 0.8
Rappel         : 0.33
F1             : 0.47
Faux positifs  : ['conversation', 'machine learning']
Skills manqués : ['product manager', 'nice', 'algorithms', 'datasets', 'mathematics computer', 'streamline', 'read', 'analytics', 'statistic', 'people manager', 'diversity and inclusion', 'patchwork', 'underwriting', 'financial service', 'statistics', 'consultant']

=== Évaluation NER — CV ===
Précision      : 0.35
Rappel         : 0.13
F1             : 0.19
Faux positifs  : ['machine', 'prototyping for', 'raw', 'github com', 'data analysis', 'solving experience', 'machine learning', 'business', 'apis', 'power', 'problem']
Skills manqués : ['datasets', 'api', 'internal business', 'analytics', 'data structuring', 'r', 'dashboards', 'b', 'french', 'automation', 'numpy', 'management', 'reliability', 'data processing', 'communication', 'ai apis', 'm', 'languages', 'raw datum', 'tableau', 'teamwork', 'power bi', 'com', 'predictive model', 'loan'